# Shorts Factory on Google Colab (free demo)

Runs the full multi-user Shorts automation app on a free Colab VM and exposes it publicly via Cloudflare Tunnel. No card or tokens needed.

**Before you start**, have ready:
- Your **Neon** pooled + direct database URLs (Neon → project → Connect)
- Optional: Telegram bot token, YouTube API key, OpenAI key, GitHub token

Then: **Runtime → Run all** (or cell-by-cell with Shift+Enter). Takes ~6–10 minutes first time.

> Free Colab sessions last ~12 hours with ~90 min idle timeout; keep this tab open while using the app.

In [ ]:
# 1. Install Node.js 20 (Next.js 16 requires >= 20.9)
!wget -q https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz -O /content/node.tar.xz
!tar -xf /content/node.tar.xz -C /content
import os
os.environ['PATH'] = '/content/node-v20.18.1-linux-x64/bin:' + os.environ['PATH']
print(os.popen('node -v && npm -v').read())

In [ ]:
# 2. Get the app code
import os, subprocess
if os.path.isdir('/content/shorts-factory'):
    print(subprocess.run(['git','-C','/content/shorts-factory','pull','--rebase'],capture_output=True,text=True).stdout or 'already up to date')
else:
    !git clone --depth 1 https://github.com/lasttopper/shorts-factory.git /content/shorts-factory
print('code ready')

In [ ]:
# 3. Configuration — fill these fields (secrets stay on this VM, never committed)
# Paste your Neon URLs from: Neon Console → Shorts-factory → Connect
DATABASE_URL = ""           # pooled URL (contains -pooler)
DATABASE_URL_UNPOOLED = ""  # direct URL (no -pooler)
AUTH_SECRET = ""            # leave empty to auto-generate one
TELEGRAM_BOT_TOKEN = ""     # optional — from @BotFather
YOUTUBE_API_KEY = ""        # optional — Google Cloud API key
OPENAI_API_KEY = ""         # optional
GITHUB_TOKEN = ""           # optional — state commits to your repo
CRON_SECRET = ""            # optional — protects /api/cron/run

import secrets
if not AUTH_SECRET:
    AUTH_SECRET = secrets.token_hex(32)
    print('GENERATED AUTH_SECRET (save this if you reuse Colab):', AUTH_SECRET)

env = f"""DATABASE_URL={DATABASE_URL}
DATABASE_URL_UNPOOLED={DATABASE_URL_UNPOOLED}
AUTH_SECRET={AUTH_SECRET}
TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN}
YOUTUBE_API_KEY={YOUTUBE_API_KEY}
OPENAI_API_KEY={OPENAI_API_KEY}
OPENAI_MODEL=gpt-4o-mini
GITHUB_TOKEN={GITHUB_TOKEN}
GITHUB_STATE_REPO=lasttopper/shorts-factory
CRON_SECRET={CRON_SECRET}
SOURCE_CHANNEL_HANDLE=@NotYourType
SHORTS_PER_RUN=10
SCHEDULE_START_HOUR=9
SLOT_INTERVAL_MIN=90
"""
open('/content/shorts-factory/.env','w').write(env)
print('config written')

In [ ]:
# 4. Install dependencies (~3–5 min on Colab)
!cd /content/shorts-factory && npm install --no-audit --no-fund --loglevel=error 2>&1 | tail -3
print('dependencies installed')

In [ ]:
# 5. Build the production app (~3–6 min)
!cd /content/shorts-factory && npm run build 2>&1 | tail -8

In [ ]:
# 6. Start the server and wait for it to become healthy
# Tables are created automatically on first request.
!cd /content/shorts-factory && nohup npm run start > /content/server.log 2>&1 &
import time, urllib.request
ok = False
for i in range(90):
    try:
        r = urllib.request.urlopen('http://127.0.0.1:3000/api/health', timeout=2)
        if r.status == 200:
            ok = True
            break
    except Exception:
        time.sleep(1)
print('SERVER HEALTHY' if ok else 'STILL STARTING — check /content/server.log')

In [ ]:
# 7. Expose it publicly with a free Cloudflare quick tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!bash -c 'nohup /content/cloudflared tunnel --url http://localhost:3000 --no-autoupdate > /content/cf.log 2>&1 &'
import time, re
url = None
for i in range(45):
    log = open('/content/cf.log').read() if __import__('os').path.exists('/content/cf.log') else ''
    m = re.search(r'https://[-a-z0-9.]+\.trycloudflare\.com', log)
    if m:
        url = m.group(0)
        break
    time.sleep(1)
print('\n==============================')
print('YOUR PUBLIC APP URL:\n', url or 'not ready yet — rerun this cell')
print('==============================')

# Use it

1. Open the printed `https://...trycloudflare.com` URL
2. **Create account** (first account = admin)
3. **Connections & Keys** → set source channel (`@NotYourType`) + your Telegram chat ID → Save → Test
4. Dashboard → **RUN TODAY'S BATCH** → watch the 10 steps; report arrives in your Telegram

**YouTube connection:** the OAuth callback auto-detects this session's tunnel URL, but the tunnel URL **changes every Colab session** — if you use *Connect with YouTube*, add the *current* tunnel URL + `/api/oauth/youtube/callback` to your Google client's Authorized redirect URIs each session, and set env var `APP_URL` to it in the Vercel/app config.

## Limits to know
- Free Colab dies after ~12h / ~90 min idle — for testing &amp; demos, not 24/7 production. Permanent home = the Vercel + Neon setup in the README.
- First request after idle can take 5–10 s (Neon + Colab wake-up).
- If the page stops responding mid-session, re-run cells **6 and 7**, then open the new URL.
- To restart everything cleanly: **Runtime → Restart and run all**.